# Lưu checkpoint C/C0 — đúng version111 và112
**Accelerator=None · Internet ON · CPU. Không chạy model hoặc notebook nguồn.**
C từ111, C0 từ112 của ducanh2403/2-vi-cai-kia-qua-nhieu.
Tải chỉ checkpoint, kiểm fullSHA đã khóa, copy nguyên byte thành C.bin và C0.bin.
Tạo Dataset Private mới, không có source/cache/500files. Cần khoảng2GiB trống.
Add-ons → Secrets: bật KAGGLE_API_TOKEN; nếu secret tên codex thì sửa SECRET_NAME.
Không dán token vào code/chat. OWNER phải đúng tài khoản sở hữu token/quyền output.


In [ ]:
from pathlib import Path
OWNER='ducanh2403'
SLUG='concept-mass-c-c0-v111-v112'
TITLE='Concept mass C C0 v111 v112 weights'
SECRET_NAME='KAGGLE_API_TOKEN'  # Đổi thành 'codex' nếu đó là tên secret của bạn
DESTINATION=Path('/kaggle/working/concept_mass_C_C0_weights_dataset')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(TITLE)<=50 or not 6<=len(SLUG)<=50:raise ValueError('TITLE/SLUG need6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    env=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_KEY','KAGGLE_USERNAME'):env.pop(key,None)
    if authenticated:env['KAGGLE_API_TOKEN']=token
    env['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError('Command failed; see redacted log. No latest/version fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)

run_cli([sys.executable,"-u","-c","import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/111', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v111', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v111; no network call.', flush=True)\n"])

run_cli([sys.executable,"-u","-c","import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/112', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v112', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v112; no network call.', flush=True)\n"])


In [ ]:
import hashlib,json
TRANSPORT_SOURCE='"""Copy only two hash-verified checkpoint files into a new flat dataset folder."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\n\nEXPECTED={\n \'C\':dict(version=111,sha256=\'99f7707f73fe5006592ea9ccbf04bce28a53a65a022cddc5c24ba6a78923c2f4\'),\n \'C0\':dict(version=112,sha256=\'5d92cd5cb48077c7332bb3ff9b295cad1df5d76ede818adabef28035f055fa9d\')}\n\ndef digest(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\') as f:\n  for chunk in iter(lambda:f.read(8*1024**2),b\'\'):h.update(chunk)\n return h.hexdigest()\n\ndef package(sources,destination,expected=EXPECTED):\n destination=Path(destination)\n if destination.exists():raise FileExistsError(\'Preserve existing dataset staging\')\n if set(sources)!=set(expected):raise ValueError(\'Need exactly C and C0 sources\')\n for name,p in sources.items():\n  if not Path(p).is_file() or digest(p)!=expected[name][\'sha256\']:raise ValueError(\'Missing/changed checkpoint \'+name)\n destination.mkdir(parents=True);files={}\n for name,p in sources.items():\n  target=destination/(name+\'.bin\');shutil.copyfile(p,target)\n  if digest(target)!=expected[name][\'sha256\']:raise ValueError(\'Copy changed checkpoint \'+name)\n  files[target.name]=dict(expected[name],bytes=target.stat().st_size)\n receipt=dict(schema=\'concept_mass_fixed_checkpoints_v1\',source_kernel=\'ducanh2403/2-vi-cai-kia-qua-nhieu\',files=files,byte_preserving=True,model_loaded=False)\n (destination/\'transport_receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\',encoding=\'utf-8\')\n return receipt\n'
assert hashlib.sha256(TRANSPORT_SOURCE.encode()).hexdigest()=='0c841acbf4bbb1330c34318aadd1d1ec046e2bceea78c1b487f8fabe3ab21165'
exec(compile(TRANSPORT_SOURCE,'package_eval_checkpoints.py','exec'))


In [ ]:
import tempfile,shutil
if DESTINATION.exists():raise FileExistsError('Preserve existing staging; check dataset page before retrying')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<2*1024**3:raise RuntimeError('Need2GiB free')
SOURCES={}
for name,info in EXPECTED.items():
    version=info['version'];kernel='ducanh2403/2-vi-cai-kia-qua-nhieu/'+str(version)
    download=Path(tempfile.mkdtemp(prefix='concept-weights-v'+str(version)+'-',dir='/tmp'))
    pattern=r'(^|/)concept_mass_'+name+r'_v2/checkpoint\.bin$'
    print('Download EXACT checkpoint:',name,kernel,flush=True)
    run_cli([sys.executable,'-m','kaggle','kernels','output',kernel,'-p',str(download),'--file-pattern',pattern])
    matches=[p for p in download.rglob('checkpoint.bin') if digest(p)==info['sha256']]
    if len(matches)!=1:raise RuntimeError(f'Version{version}: {len(matches)} matching checkpoints. STOP; no latest fallback. Check version output files.')
    SOURCES[name]=matches[0]
receipt=package(SOURCES,DESTINATION)
print('Verified both full checkpoint hashes; ready for private dataset upload.',flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata=DESTINATION/'dataset-metadata.json'
if metadata.exists():raise FileExistsError('Preserve existing metadata')
try:
    metadata.write_text(json.dumps(dict(title=TITLE,id=OWNER+'/'+SLUG,licenses=[dict(name='other')],description='Private research trained weights. Exact C endpoint500 Output111 and C0 endpoint500 Output112, byte-preserving bin files. Upstream data/model rights retained. See transport_receipt.json. No new training or inference.'),indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata.unlink(missing_ok=True)
print('Creation accepted; processing may still be pending. Do not repeatedly create.',flush=True)
print('https://www.kaggle.com/datasets/'+OWNER+'/'+SLUG,flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',OWNER+'/'+SLUG])
except RuntimeError:print('Status not confirmed; check the dataset page before retrying.',flush=True)


Đợi dataset xử lý xong, có C.bin, C0.bin và transport_receipt.json ở gốc.
Add dataset vào notebook đánh giá. Default path:
/kaggle/input/datasets/ducanh2403/concept-mass-c-c0-v111-v112
Giữ weights; notebook này chưa được assistant chạy API.
